In [ ]:
from typing import Optional
import matplotlib.pyplot as plt
import numpy as np
from ase import Atoms


kB_eV = 8.617333262145e-5  # eV/K


def bcc_supercell_fractional(n: int) -> np.ndarray:
    basis = np.array(
        [
            [0.0, 0.0, 0.0],
            [0.5, 0.5, 0.5],
        ],
        dtype=float,
    )

    pos = []
    for i in range(n):
        for j in range(n):
            for k in range(n):
                cell_index = np.array([i, j, k], dtype=float)
                for basis_atom in basis:
                    pos.append((cell_index + basis_atom) / n)
    return np.array(pos, dtype=float)


def fcc_supercell_fractional(n: int) -> np.ndarray:
    basis = np.array(
        [
            [0.0, 0.0, 0.0],
            [0.0, 0.5, 0.5],
            [0.5, 0.0, 0.5],
            [0.5, 0.5, 0.0],
        ],
        dtype=float,
    )

    pos = []
    for i in range(n):
        for j in range(n):
            for k in range(n):
                cell_index = np.array([i, j, k], dtype=float)
                for basis_atom in basis:
                    pos.append((cell_index + basis_atom) / n)
    return np.array(pos, dtype=float)


def hcp_supercell_fractional(nx: int, ny: int, nz: int) -> np.ndarray:
    basis = np.array(
        [
            [0.0, 0.0, 0.0],
            [2.0 / 3.0, 1.0 / 3.0, 0.5],
        ],
        dtype=float,
    )

    pos = []
    scale = np.array([nx, ny, nz], dtype=float)
    for i in range(nx):
        for j in range(ny):
            for k in range(nz):
                cell_index = np.array([i, j, k], dtype=float)
                for basis_atom in basis:
                    pos.append((cell_index + basis_atom) / scale)
    return np.array(pos, dtype=float)


def cubic_cell(a: float, n: int) -> np.ndarray:
    return np.array(
        [
            [n * a, 0.0, 0.0],
            [0.0, n * a, 0.0],
            [0.0, 0.0, n * a],
        ],
        dtype=float,
    )


def hcp_cell(a: float, c: float, nx: int, ny: int, nz: int) -> np.ndarray:
    a1 = np.array([a, 0.0, 0.0])
    a2 = np.array([0.5 * a, np.sqrt(3.0) * a / 2.0, 0.0])
    a3 = np.array([0.0, 0.0, c])
    return np.array(
        [
            nx * a1,
            ny * a2,
            nz * a3,
        ],
        dtype=float,
    )


def sigma_from_temperature(T: float, k_eff: float = 10.0) -> float:
    if T < 0:
        raise ValueError("Temperature must be non-negative.")
    if k_eff <= 0:
        raise ValueError("k_eff must be positive.")
    return np.sqrt(kB_eV * T / k_eff)


def generate_thermal_displacements(
    N: int,
    T: float,
    k_eff: float = 10.0,
    seed=None,
):
    rng = np.random.default_rng(seed)
    sigma = sigma_from_temperature(T, k_eff)

    dx = rng.normal(0.0, sigma, N)
    dy = rng.normal(0.0, sigma, N)
    dz = rng.normal(0.0, sigma, N)

    disp = np.column_stack((dx, dy, dz))
    r = np.sqrt(dx**2 + dy**2 + dz**2)
    return disp, r, dx, dy, dz, sigma


def generate_paramagnetic_spins_zero_net(
    N: int,
    m_abs: float = 0.35,
    seed=None,
    pair_opposites: bool = True,
):
    if N < 2:
        raise ValueError("N must be at least 2 for a paramagnetic-like zero-net configuration.")
    if m_abs < 0:
        raise ValueError("m_abs must be non-negative.")

    rng = np.random.default_rng(seed)

    if pair_opposites and (N % 2 == 0):
        n_half = N // 2
        vec = rng.normal(size=(n_half, 3))
        norms = np.linalg.norm(vec, axis=1)
        bad = norms < 1e-14
        while np.any(bad):
            vec[bad] = rng.normal(size=(np.sum(bad), 3))
            norms = np.linalg.norm(vec, axis=1)
            bad = norms < 1e-14
        n_hat = vec / norms[:, None]
        n_hat = np.vstack([n_hat, -n_hat])
        rng.shuffle(n_hat, axis=0)
    else:
        n_hat = rng.normal(size=(N, 3))
        norms = np.linalg.norm(n_hat, axis=1)
        bad = norms < 1e-14
        while np.any(bad):
            n_hat[bad] = rng.normal(size=(np.sum(bad), 3))
            norms = np.linalg.norm(n_hat, axis=1)
            bad = norms < 1e-14
        n_hat = n_hat / norms[:, None]

        for _ in range(100):
            n_hat = n_hat - np.mean(n_hat, axis=0, keepdims=True)
            norms = np.linalg.norm(n_hat, axis=1)
            bad = norms < 1e-14
            if np.any(bad):
                n_hat[bad] = rng.normal(size=(np.sum(bad), 3))
                norms = np.linalg.norm(n_hat, axis=1)
            n_hat = n_hat / norms[:, None]

    m_cart = m_abs * n_hat
    nx = n_hat[:, 0]
    ny = n_hat[:, 1]
    nz = n_hat[:, 2]

    angle1 = np.degrees(np.arccos(np.clip(nz, -1.0, 1.0)))
    angle2 = np.degrees(np.arctan2(ny, nx))
    angle2 = np.where(angle2 < 0.0, angle2 + 360.0, angle2)

    theta_deg = angle1.copy()
    net_m = np.sum(m_cart, axis=0)
    return m_cart, angle1, angle2, theta_deg, net_m


def write_qe_positions_txt(
    frac_pos: np.ndarray,
    path: str,
    species_labels=None,
    species: str = "Fe",
) -> None:
    with open(path, "w", encoding="utf-8") as f:
        f.write("ATOMIC_POSITIONS (crystal)\n")
        if species_labels is None:
            for x, y, z in frac_pos:
                f.write(f"{species} {x:.10f} {y:.10f} {z:.10f}\n")
        else:
            for label, (x, y, z) in zip(species_labels, frac_pos):
                f.write(f"{label} {x:.10f} {y:.10f} {z:.10f}\n")


def write_displacements_txt(disp_cart: np.ndarray, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        f.write("# dx dy dz  (Angstrom)\n")
        for dx, dy, dz in disp_cart:
            f.write(f"{dx:.10f} {dy:.10f} {dz:.10f}\n")


def write_spin_vectors_txt(m_cart: np.ndarray, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        f.write("# mx my mz\n")
        for mx, my, mz in m_cart:
            f.write(f"{mx:.10f} {my:.10f} {mz:.10f}\n")


def write_qe_spin_parameters_txt(
    angle1: np.ndarray,
    angle2: np.ndarray,
    path: str,
    m_abs: float = 0.5,
) -> None:
    with open(path, "w", encoding="utf-8") as f:
        f.write("# QE noncollinear initial magnetic parameters\n")
        for i in range(len(angle1)):
            f.write(f"starting_magnetization({i + 1}) = {m_abs:.10f}\n")
            f.write(f"angle1({i + 1}) = {angle1[i]:.10f}\n")
            f.write(f"angle2({i + 1}) = {angle2[i]:.10f}\n")


def write_qe_input(
    frac_pos: np.ndarray,
    cell: np.ndarray,
    out_path: str,
    pseudo_file: str = "Fe.pbe-spn-kjpaw_psl.1.0.0.UPF",
    mass: float = 55.845,
    prefix: str = "Fe_input",
    ecutwfc: float = 70.0,
    ecutrho: float = 560.0,
    degauss: float = 0.02,
    kpts=(2, 2, 2, 0, 0, 0),
    calculation: str = "scf",
    species_base: str = "Fe",
    pseudo_dir: str = "../pseudo/",
    outdir: str = "./tmp/",
    spin_mode: str = "none",
    angle1: Optional[np.ndarray] = None,
    angle2: Optional[np.ndarray] = None,
    m_abs: float = 0.0,
    lspinorb: bool = False,
    mixing_beta: float = 0.01,
    electron_maxstep: int = 400,
    conv_thr: float = 1.0e-4,
    diagonalization: str = "david",
    mixing_mode: str = "local-TF",
    mixing_ndim: int = 8,
    use_constraints: bool = False,
    constraint_kind: str = "atomic direction",
    lambda_value: float = 0.1,
) -> None:
    if spin_mode not in {"none", "paramagnetic"}:
        raise ValueError("spin_mode must be 'none' or 'paramagnetic'.")

    N = len(frac_pos)
    use_noncollinear = spin_mode == "paramagnetic"

    if use_noncollinear:
        if angle1 is None or angle2 is None:
            raise ValueError("angle1 and angle2 are required for paramagnetic QE input.")
        if len(angle1) != N or len(angle2) != N:
            raise ValueError("angle arrays must have one entry per atom.")
        species_labels = [f"{species_base}{i + 1}" for i in range(N)]
        ntyp = N
    else:
        species_labels = [species_base] * N
        ntyp = 1

    with open(out_path, "w", encoding="utf-8") as f:
        f.write("&CONTROL\n")
        f.write(f"    calculation   = '{calculation}'\n")
        f.write("    restart_mode  = 'from_scratch'\n")
        f.write(f"    prefix        = '{prefix}'\n")
        f.write(f"    pseudo_dir    = '{pseudo_dir}'\n")
        f.write(f"    outdir        = '{outdir}'\n")
        f.write("    etot_conv_thr = 1.0d-4\n")
        f.write("    forc_conv_thr = 1.0d-4\n")
        f.write("    tstress       = .true.\n")
        f.write("    tprnfor       = .true.\n")
        f.write("/\n\n")

        f.write("&SYSTEM\n")
        f.write("    ibrav = 0\n")
        f.write(f"    nat   = {N}\n")
        f.write(f"    ntyp  = {ntyp}\n")
        f.write(f"    ecutwfc     = {ecutwfc}\n")
        f.write(f"    ecutrho     = {ecutrho}\n")
        f.write("    occupations = 'smearing'\n")
        f.write("    smearing    = 'mv'\n")
        f.write(f"    degauss     = {degauss}\n")

        if use_noncollinear:
            f.write("    noncolin    = .true.\n")
            f.write(f"    lspinorb    = {'.true.' if lspinorb else '.false.'}\n")

            if use_constraints:
                f.write(f"    constrained_magnetization = '{constraint_kind}'\n")
                f.write(f"    lambda = {lambda_value}\n")

            for i in range(N):
                f.write(f"    starting_magnetization({i + 1}) = {m_abs:.10f}\n")
                f.write(f"    angle1({i + 1}) = {angle1[i]:.10f}\n")
                f.write(f"    angle2({i + 1}) = {angle2[i]:.10f}\n")
        f.write("/\n\n")

        f.write("&ELECTRONS\n")
        f.write(f"    conv_thr         = {conv_thr:.1e}\n")
        f.write(f"    mixing_beta      = {mixing_beta}\n")
        f.write(f"    electron_maxstep = {electron_maxstep}\n")
        f.write(f"    diagonalization  = '{diagonalization}'\n")
        f.write(f"    mixing_mode      = '{mixing_mode}'\n")
        f.write(f"    mixing_ndim      = {mixing_ndim}\n")
        f.write("/\n\n")

        if calculation in ["relax", "vc-relax"]:
            f.write("&IONS\n")
            f.write("    ion_dynamics = 'bfgs'\n")
            f.write("/\n\n")

        f.write("ATOMIC_SPECIES\n")
        if use_noncollinear:
            for label in species_labels:
                f.write(f"{label}  {mass:.3f}  {pseudo_file}\n")
        else:
            f.write(f"{species_base}  {mass:.3f}  {pseudo_file}\n")
        f.write("\n")

        f.write("ATOMIC_POSITIONS (crystal)\n")
        for label, (x, y, z) in zip(species_labels, frac_pos):
            f.write(f"{label} {x:.10f} {y:.10f} {z:.10f}\n")
        f.write("\n")

        f.write("CELL_PARAMETERS angstrom\n")
        for row in cell:
            f.write(f"{row[0]:.12f} {row[1]:.12f} {row[2]:.12f}\n")
        f.write("\n")

        f.write("K_POINTS automatic\n")
        f.write(f"{kpts[0]} {kpts[1]} {kpts[2]} {kpts[3]} {kpts[4]} {kpts[5]}\n")


def build_ase_atoms_from_fractional(
    frac_pos: np.ndarray,
    cell: np.ndarray,
    symbol: str = "Fe",
) -> Atoms:
    symbols = [symbol] * len(frac_pos)
    return Atoms(symbols=symbols, scaled_positions=frac_pos, cell=cell, pbc=True)


def rotation_matrix_x(angle_deg):
    angle_rad = np.radians(angle_deg)
    return np.array(
        [
            [1, 0, 0],
            [0, np.cos(angle_rad), -np.sin(angle_rad)],
            [0, np.sin(angle_rad), np.cos(angle_rad)],
        ],
        dtype=float,
    )


def rotation_matrix_y(angle_deg):
    angle_rad = np.radians(angle_deg)
    return np.array(
        [
            [np.cos(angle_rad), 0, np.sin(angle_rad)],
            [0, 1, 0],
            [-np.sin(angle_rad), 0, np.cos(angle_rad)],
        ],
        dtype=float,
    )


def rotation_matrix_z(angle_deg):
    angle_rad = np.radians(angle_deg)
    return np.array(
        [
            [np.cos(angle_rad), -np.sin(angle_rad), 0],
            [np.sin(angle_rad), np.cos(angle_rad), 0],
            [0, 0, 1],
        ],
        dtype=float,
    )


def build_rotation_matrix(rotation: str = "35x,45y,0z") -> np.ndarray:
    R = np.eye(3)
    for item in rotation.split(","):
        item = item.strip()
        if item.endswith("x"):
            R = rotation_matrix_x(float(item[:-1])) @ R
        elif item.endswith("y"):
            R = rotation_matrix_y(float(item[:-1])) @ R
        elif item.endswith("z"):
            R = rotation_matrix_z(float(item[:-1])) @ R
    return R


def plot_spin_angles(theta_deg: np.ndarray, angle1: np.ndarray, angle2: np.ndarray):
    atom_index = np.arange(1, len(theta_deg) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(atom_index, theta_deg, "o-")
    plt.xlabel("Atom index")
    plt.ylabel("Canting angle from +z (deg)")
    plt.title("Initial paramagnetic spin angles")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.hist(theta_deg, bins=20, edgecolor="black", alpha=0.7)
    plt.xlabel("Canting angle from +z (deg)")
    plt.ylabel("Count")
    plt.title("Distribution of spin angles")
    plt.tight_layout()
    plt.show()


def plot_atoms_with_spins_2d_manual(
    atoms: Atoms,
    spins_cart: Optional[np.ndarray],
    ax,
    rotation: str = "35x,45y,0z",
    circle_scale: float = 0.35,
    arrow_scale: float = 1.2,
    arrow_width: float = 0.012,
    title: Optional[str] = None,
    draw_cell: bool = True,
):
    pos = atoms.get_positions()
    cell = atoms.get_cell().array
    R = build_rotation_matrix(rotation)

    pos_rot = pos @ R.T
    x = pos_rot[:, 0]
    y = pos_rot[:, 1]

    if spins_cart is not None:
        spin_rot = spins_cart @ R.T
        u = spin_rot[:, 0]
        v = spin_rot[:, 1]
    else:
        u = None
        v = None

    if len(pos) > 1:
        dmin = np.min(
            [
                np.linalg.norm(pos[i] - pos[j])
                for i in range(len(pos))
                for j in range(i + 1, len(pos))
            ]
        )
        radius = circle_scale * dmin
    else:
        radius = 0.5

    if draw_cell:
        origin = np.array([0.0, 0.0, 0.0]) @ R.T
        a1 = cell[0] @ R.T
        a2 = cell[1] @ R.T
        a3 = cell[2] @ R.T

        corners = np.array(
            [
                origin,
                a1,
                a2,
                a3,
                a1 + a2,
                a1 + a3,
                a2 + a3,
                a1 + a2 + a3,
            ]
        )

        edges = [
            (0, 1),
            (0, 2),
            (0, 3),
            (1, 4),
            (1, 5),
            (2, 4),
            (2, 6),
            (3, 5),
            (3, 6),
            (4, 7),
            (5, 7),
            (6, 7),
        ]

        for i, j in edges:
            ax.plot(
                [corners[i, 0], corners[j, 0]],
                [corners[i, 1], corners[j, 1]],
                linestyle=(0, (8, 8)),
                color="black",
                linewidth=1.0,
                alpha=0.9,
                zorder=1,
            )

    order = np.argsort(pos_rot[:, 2])
    for idx in order:
        circle = plt.Circle(
            (x[idx], y[idx]),
            radius=radius,
            facecolor="#e36a2e",
            edgecolor="black",
            linewidth=1.0,
            zorder=3,
        )
        ax.add_patch(circle)

    if u is not None and v is not None:
        ax.quiver(
            x,
            y,
            arrow_scale * u,
            arrow_scale * v,
            angles="xy",
            scale_units="xy",
            scale=1,
            color="black",
            width=arrow_width,
            headwidth=3.5,
            headlength=5.0,
            headaxislength=4.5,
            zorder=4,
        )

    ax.set_aspect("equal")
    ax.set_xlabel("X (A)")
    ax.set_ylabel("Y (A)")

    xmin, xmax = np.min(x), np.max(x)
    ymin, ymax = np.min(y), np.max(y)
    padx = 0.9 * radius + 0.5
    pady = 0.9 * radius + 0.5
    ax.set_xlim(xmin - padx, xmax + padx)
    ax.set_ylim(ymin - pady, ymax + pady)

    if title is not None:
        ax.set_title(title)



############################################            ############################################
############################################            ############################################
#########                                   Parameters                                     #########
############################################            ############################################
############################################            ############################################

if __name__ == "__main__":
    structure = "hcp"  # "bcc", "fcc", "hcp"
    n = 4
    nx, ny, nz = 4, 4, 4

    a = 2.14
    c = 3.40

    T = 0.0
    k_eff = 8.0
    seed_disp = 42

    # "none" -> structural QE input only
    # "paramagnetic" -> noncollinear paramagnetic QE input only
    # "both" -> write both QE inputs in one run
    qe_spin_mode = "none"

    m_abs = 0.35
    seed_spin = 123
    pair_opposites = True

    pseudo_file = "Fe.pbe-spn-kjpaw_psl.1.0.0.UPF"
    species_base = "Fe"

    calculation = "relax"
    ecutwfc = 70.0
    ecutrho = 560.0
    degauss = 0.03
    kpts = (4, 4, 4, 0, 0, 0)

    mixing_beta = 0.03
    electron_maxstep = 400
    conv_thr = 1.0e-6
    diagonalization = "cg"
    mixing_mode = "local-TF"
    mixing_ndim = 8

    use_constraints = True
    constraint_kind = "atomic direction"
    lambda_value = 0.2

    show_spin_plots = False
    show_spin_structure = False

    if qe_spin_mode not in {"none", "paramagnetic", "both"}:
        raise ValueError("qe_spin_mode must be 'none', 'paramagnetic', or 'both'.")

    if structure.lower() == "bcc":
        frac_pos = bcc_supercell_fractional(n)
        cell = cubic_cell(a, n)
        structure_tag = f"bcc_{n}x{n}x{n}"
    elif structure.lower() == "fcc":
        frac_pos = fcc_supercell_fractional(n)
        cell = cubic_cell(a, n)
        structure_tag = f"fcc_{n}x{n}x{n}"
    elif structure.lower() == "hcp":
        frac_pos = hcp_supercell_fractional(nx, ny, nz)
        cell = hcp_cell(a, c, nx, ny, nz)
        structure_tag = f"hcp_{nx}x{ny}x{nz}"
    else:
        raise ValueError("structure must be 'bcc', 'fcc', or 'hcp'")

    N = len(frac_pos)

    disp_cart, r_disp, dx, dy, dz, sigma_disp = generate_thermal_displacements(
        N=N,
        T=T,
        k_eff=k_eff,
        seed=seed_disp,
    )

    cell_inv = np.linalg.inv(cell)
    disp_frac = disp_cart @ cell_inv
    frac_pos_displaced = (frac_pos + disp_frac) % 1.0

    atoms_original = build_ase_atoms_from_fractional(frac_pos, cell=cell, symbol=species_base)
    atoms_displaced = build_ase_atoms_from_fractional(frac_pos_displaced, cell=cell, symbol=species_base)

    pos_file = f"{species_base}_{structure_tag}_positions_displaced_{int(T)}K.txt"
    disp_file = f"{species_base}_{structure_tag}_displacements_{int(T)}K.txt"

    write_qe_positions_txt(frac_pos_displaced, pos_file)
    write_displacements_txt(disp_cart, disp_file)

    requested_qe_modes = ["none", "paramagnetic"] if qe_spin_mode == "both" else [qe_spin_mode]
    written_qe_inputs = []

    m_cart = None
    angle1 = None
    angle2 = None
    theta_deg = None
    net_m = None

    if "none" in requested_qe_modes:
        qe_input_no_spin = f"{species_base}_{structure_tag}_no_spin_{int(T)}K.in"
        write_qe_input(
            frac_pos=frac_pos_displaced,
            cell=cell,
            out_path=qe_input_no_spin,
            pseudo_file=pseudo_file,
            mass=55.845,
            prefix=f"{species_base}_{structure_tag}_no_spin",
            ecutwfc=ecutwfc,
            ecutrho=ecutrho,
            degauss=degauss,
            kpts=kpts,
            calculation=calculation,
            species_base=species_base,
            pseudo_dir="../pseudo/",
            outdir="./tmp/",
            spin_mode="none",
            mixing_beta=mixing_beta,
            electron_maxstep=electron_maxstep,
            conv_thr=conv_thr,
            diagonalization=diagonalization,
            mixing_mode=mixing_mode,
            mixing_ndim=mixing_ndim,
        )
        written_qe_inputs.append(qe_input_no_spin)

    if "paramagnetic" in requested_qe_modes:
        m_cart, angle1, angle2, theta_deg, net_m = generate_paramagnetic_spins_zero_net(
            N=N,
            m_abs=m_abs,
            seed=seed_spin,
            pair_opposites=pair_opposites,
        )

        spin_vec_file = f"{species_base}_{structure_tag}_spin_vectors_paramagnetic_m{m_abs:.2f}.txt"
        spin_qe_file = f"{species_base}_{structure_tag}_qe_spin_params_paramagnetic_m{m_abs:.2f}.txt"
        qe_input_para = f"{species_base}_{structure_tag}_noncollinear_paramagnetic_{int(T)}K.in"

        write_spin_vectors_txt(m_cart, spin_vec_file)
        write_qe_spin_parameters_txt(angle1, angle2, spin_qe_file, m_abs=m_abs)

        write_qe_input(
            frac_pos=frac_pos_displaced,
            cell=cell,
            out_path=qe_input_para,
            pseudo_file=pseudo_file,
            mass=55.845,
            prefix=f"{species_base}_{structure_tag}_noncollinear_paramagnetic",
            ecutwfc=ecutwfc,
            ecutrho=ecutrho,
            degauss=degauss,
            kpts=kpts,
            calculation=calculation,
            species_base=species_base,
            pseudo_dir="../pseudo/",
            outdir="./tmp/",
            spin_mode="paramagnetic",
            angle1=angle1,
            angle2=angle2,
            m_abs=m_abs,
            lspinorb=False,
            mixing_beta=mixing_beta,
            electron_maxstep=electron_maxstep,
            conv_thr=conv_thr,
            diagonalization=diagonalization,
            mixing_mode=mixing_mode,
            mixing_ndim=mixing_ndim,
            use_constraints=use_constraints,
            constraint_kind=constraint_kind,
            lambda_value=lambda_value,
        )
        written_qe_inputs.append(qe_input_para)
    else:
        spin_vec_file = None
        spin_qe_file = None

    print("=========================================================")
    print(f"Structure                = {structure_tag}")
    print(f"nat                      = {N}")
    print(f"Temperature              = {T} K")
    print(f"k_eff                    = {k_eff} eV/Angstrom^2")
    print(f"sigma_disp               = {sigma_disp:.6f} Angstrom")
    print(f"Mean |displacement|      = {np.mean(r_disp):.6f} Angstrom")
    print(f"Max  |displacement|      = {np.max(r_disp):.6f} Angstrom")
    print("---------------------------------------------------------")
    print(f"qe_spin_mode             = {qe_spin_mode}")

    if m_cart is not None and theta_deg is not None and net_m is not None:
        print("Paramagnetic initialization")
        print(f"Net moment x             = {net_m[0]:.12e}")
        print(f"Net moment y             = {net_m[1]:.12e}")
        print(f"Net moment z             = {net_m[2]:.12e}")
        print(f"|Net moment|             = {np.linalg.norm(net_m):.12e}")
        print(f"m_abs                    = {m_abs}")
        print(f"Mean canting angle       = {np.mean(theta_deg):.6f} deg")
        print(f"Max  canting angle       = {np.max(theta_deg):.6f} deg")
        print(f"use_constraints          = {use_constraints}")
    else:
        print("Paramagnetic initialization = disabled")

    print("---------------------------------------------------------")
    print(f"QE calculation           = {calculation}")
    print(f"mixing_beta              = {mixing_beta}")
    print(f"mixing_mode              = {mixing_mode}")
    print(f"mixing_ndim              = {mixing_ndim}")
    print(f"diagonalization          = {diagonalization}")
    print("---------------------------------------------------------")
    print(f"Wrote: {pos_file}")
    print(f"Wrote: {disp_file}")
    if spin_vec_file is not None:
        print(f"Wrote: {spin_vec_file}")
    if spin_qe_file is not None:
        print(f"Wrote: {spin_qe_file}")
    for qe_input_file in written_qe_inputs:
        print(f"Wrote: {qe_input_file}")
    print("=========================================================")

    if show_spin_structure:
        fig, axes = plt.subplots(1, 2, figsize=(14, 7))

        plot_atoms_with_spins_2d_manual(
            atoms_original,
            m_cart,
            ax=axes[0],
            rotation="0x,0y,0z",
            circle_scale=0.18,
            arrow_scale=10.0,
            arrow_width=0.006,
            title="Original",
            draw_cell=True,
        )

        plot_atoms_with_spins_2d_manual(
            atoms_displaced,
            m_cart,
            ax=axes[1],
            rotation="0x,0y,0z",
            circle_scale=0.18,
            arrow_scale=10.0,
            arrow_width=0.006,
            title=f"Displaced at {int(T)} K",
            draw_cell=True,
        )

        plt.tight_layout()
        if m_cart is not None:
            plot_file = f"{species_base}_{structure_tag}_paramagnetic_spins_{int(T)}K.png"
        else:
            plot_file = f"{species_base}_{structure_tag}_structure_{int(T)}K.png"
        plt.savefig(plot_file, dpi=300, bbox_inches="tight")
        print(f"Wrote: {plot_file}")
        plt.show()

    if m_cart is not None and show_spin_plots:
        plot_spin_angles(theta_deg, angle1, angle2)
